In [1]:
import polars as pl
import pathlib
import re
import requests
from requests.adapters import HTTPAdapter, Retry
from tqdm.auto import tqdm
import json

from Bio import Align
from Bio.Align import substitution_matrices

retries = Retry(total=5, backoff_factor=0.25, status_forcelist=[500, 502, 503, 504])
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retries))

In [ ]:


# # CD33 signal peptide: exact 20-aa sequence used in all constructs (Tegel et al.)
# # (MPLLLLLPLLWAGALAMA = base 18 aa + AAA linker = 20 aa total)
# CD33_SP_SEQ  = 'MPLLLLLPLLWAGALAMAAA'
# CD33_SP_ALT  = 'MPLLLLLPLLWAGALAMA'     # fallback for shorter variants

# # Average amino acid residue masses (Da), used for MW estimation
# AA_RESIDUE_MW = {
#     'A': 71.08,  'R': 156.19, 'N': 114.10, 'D': 115.09, 'C': 103.14,
#     'E': 129.12, 'Q': 128.13, 'G': 57.05,  'H': 137.14, 'I': 113.16,
#     'L': 113.16, 'K': 128.17, 'M': 131.20, 'F': 147.18, 'P': 97.12,
#     'S': 87.08,  'T': 101.10, 'W': 186.21, 'Y': 163.18, 'V': 99.13,
# }

# Module-level aligner: created once and reused
_pwaligner: Align.PairwiseAligner | None = None


def _get_aligner() -> Align.PairwiseAligner:
    global _pwaligner
    if _pwaligner is None:
        _pwaligner = Align.PairwiseAligner()
        _pwaligner.mode = 'local'
        _pwaligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
        _pwaligner.open_gap_score = -11
        _pwaligner.extend_gap_score = -1
    return _pwaligner

def find_boundaries_by_alignment(
    target: str,
    canonical: str,
    score_fraction: float = 0.7,
    min_block_len: int = 5,
    max_alignments: int = 100,
) -> tuple[int, int] | None:
    """Find construct boundaries in the canonical UniProt sequence via BioPython local alignment.

    Iterates over all high-scoring local alignments (score ≥ score_fraction × best_score)
    and collects every aligned block in canonical coordinates.  Blocks from all alignments
    are merged into a single union interval so that, for example, two fragments covering
    canonical positions [5-10] and [15-20] return (5, 20).

    Parameters
    ----------
    target        : stripped construct sequence (CD33SP and TEV-HPC4 already removed)
    canonical     : full-length canonical UniProt sequence
    score_fraction: minimum score relative to the best alignment to include an alignment
    min_block_len : minimum aligned block length (in canonical residues) to record
    max_alignments: maximum number of alignments to inspect

    Returns
    -------
    (start, end) — 1-based, inclusive — covering the union of all aligned canonical blocks;
    None if no alignment with a positive score is found.
    """
    if not target or not canonical:
        return None

    aligner = _get_aligner()
    alignments = aligner.align(canonical, target)

    best_score: float | None = None
    canon_starts: list[int] = []
    canon_ends:   list[int] = []

    for i, aln in enumerate(alignments):
        if i >= max_alignments:
            break

        score = aln.score
        if best_score is None:
            if score <= 0:
                return None
            best_score = score
        elif score < score_fraction * best_score:
            # alignments come in descending score order; stop when below threshold
            break

        # aln.coordinates: shape (2, N+1)
        #   row 0 — positions in canonical (the "subject" sequence passed first to aligner)
        #   row 1 — positions in target (the "query")
        # Each consecutive pair (j, j+1) describes one block:
        #   if both canonical and target advance → aligned block
        #   if only canonical advances → gap in target (deletion from canonical)
        #   if only target advances    → gap in canonical (insertion in target)
        coords     = aln.coordinates
        canon_pos  = coords[0]
        target_pos = coords[1]

        for j in range(len(canon_pos) - 1):
            c_start, c_end = int(canon_pos[j]),  int(canon_pos[j + 1])
            t_start, t_end = int(target_pos[j]), int(target_pos[j + 1])
            # Both sequences must advance and the block must be long enough
            if c_end > c_start and t_end > t_start and (c_end - c_start) >= min_block_len:
                canon_starts.append(c_start)
                canon_ends.append(c_end)

    if not canon_starts:
        return None

    # 1-based, inclusive union over all collected blocks
    return min(canon_starts) + 1, max(canon_ends)




In [3]:

az_report_file = pathlib.Path('~/data/secretome/AZ_report_20181221.xlsx').expanduser()

az_df = (
    pl.read_excel(az_report_file, sheet_name='az_production_report_181221')
    .rename({
        'Virtual construct':                      'vc_id',
        'Gene name':                              'gene_name',
        'Uniprot ID':                             'uniprot_id',
        'Concentration [µM]':                     'concentration_um',
        'Amount delivered [nmol]':                'amount_nmol',
        'Host':                                   'host',
        'Construct seq [aa] (CD33_Target_TEV_HPC4)': 'aa_seq',
        'Status':                                 'status',
        'Comment':                                'comment',
        'Recloning at KTH possible':              'recloning_kth',
        'Secretome annotation':                   'secretome_annotation',
    })
    .with_columns(
        pl.col('concentration_um').cast(pl.Float64, strict=False),
        pl.col('amount_nmol').cast(pl.Float64, strict=False),
        # target_region=pl.col('aa_seq').map_elements(parse_target_region, return_dtype=pl.String),
    )
    .with_columns(
        # Estimate production scale from total amount delivered
        scale=pl.when(pl.col('host') == 'HEK')
               .then(pl.lit('HEK_small'))
               .when(pl.col('amount_nmol') < 30)
               .then(pl.lit('CHO_small'))
               .when(pl.col('amount_nmol') < 200)
               .then(pl.lit('CHO_medium'))
               .otherwise(pl.lit('CHO_pilot')),
    )
)

print(f"Total records: {len(az_df)}")
print(f"Unique VCs: {az_df['vc_id'].n_unique()}")
print(f"Status counts: {az_df['status'].value_counts().sort('status')}")
print(f"Host counts: {az_df['host'].value_counts()}")
az_df.head(3)


Total records: 3605
Unique VCs: 3069
Status counts: shape: (3, 2)
┌───────────────────┬───────┐
│ status            ┆ count │
│ ---               ┆ ---   │
│ str               ┆ u32   │
╞═══════════════════╪═══════╡
│ Delivered         ┆ 2154  │
│ Failed production ┆ 1381  │
│ In production     ┆ 70    │
└───────────────────┴───────┘
Host counts: shape: (2, 2)
┌──────┬───────┐
│ host ┆ count │
│ ---  ┆ ---   │
│ str  ┆ u32   │
╞══════╪═══════╡
│ HEK  ┆ 189   │
│ CHO  ┆ 3416  │
└──────┴───────┘


vc_id,gene_name,uniprot_id,concentration_um,amount_nmol,host,aa_seq,status,comment,recloning_kth,secretome_annotation,scale
str,str,str,f64,f64,str,str,str,str,str,str,str
"""VC000001""","""APOE""","""P02649""",6.1,29.6,"""CHO""","""MPLLLLLPLLWAGALAMAAAKVEQAVETEP…","""Delivered""",null,null,"""Secreted to blood""","""CHO_small"""
"""VC000002""","""C1QC""","""P02747""",6.5,17.9,"""CHO""","""MPLLLLLPLLWAGALAMAAAANTGCYGIPG…","""Delivered""",null,null,"""Secreted to blood""","""CHO_small"""
"""VC000003""","""ECM1""","""Q16610""",8.0,30.2,"""CHO""","""MPLLLLLPLLWAGALAMAAASEGGFTATGQ…","""Delivered""","""Reorder by AZ""",null,"""Locally secreted in extracellu…","""CHO_medium"""


In [4]:
az_df.filter(host='HEK')

vc_id,gene_name,uniprot_id,concentration_um,amount_nmol,host,aa_seq,status,comment,recloning_kth,secretome_annotation,scale
str,str,str,f64,f64,str,str,str,str,str,str,str
"""VC000005""","""FGF22""","""Q9HCT0""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMGTPSASRGPRSYP…","""Failed production""",null,"""X""","""Locally secreted""","""HEK_small"""
"""VC000010""","""CCDC80""","""Q76M96""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMPHPHATIRGSHGG…","""Failed production""",null,"""X""","""Locally secreted in extracellu…","""HEK_small"""
"""VC000014""","""CTGF""","""P29279""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMQNCSGPCRCPDEP…","""In production""",null,"""X""","""Locally secreted""","""HEK_small"""
"""VC000036""","""LRRC17""","""Q8N6Y2""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMAELRKASPGSVRS…","""Failed production""",null,"""X""","""Locally secreted""","""HEK_small"""
"""VC000045""","""THBS1""","""P07996""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMRIPESGGDNSVFD…","""In production""",null,"""X""","""Locally secreted in extracellu…","""HEK_small"""
…,…,…,…,…,…,…,…,…,…,…,…
"""VC003126""","""RELT""","""Q969Z4""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMSTTLWQCPPGEEP…","""Failed production""",null,"""X""",null,"""HEK_small"""
"""VC003132""","""SUSD2""","""Q9UGT4""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMQESCSMRCGALDG…","""Failed production""",null,"""X""",null,"""HEK_small"""
"""VC003174""","""INSR""","""P06213""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMGEVCPGMDIRNNL…","""Failed production""",null,"""X""",null,"""HEK_small"""


In [5]:
az_df.filter(uniprot_id='Q76M96')

vc_id,gene_name,uniprot_id,concentration_um,amount_nmol,host,aa_seq,status,comment,recloning_kth,secretome_annotation,scale
str,str,str,f64,f64,str,str,str,str,str,str,str
"""VC000010""","""CCDC80""","""Q76M96""",null,null,"""HEK""","""MPLLLLLPLLWAGALAMPHPHATIRGSHGG…","""Failed production""",null,"""X""","""Locally secreted in extracellu…","""HEK_small"""


In [ ]:
uniprot_cache_file = pathlib.Path(
    '~/Documents/nicola_paper_2026/biostudies/tegel_secretome_uniprot.parquet'
).expanduser()


def get_uniprot_data(uniprot_id: str) -> dict:
    """Fetch protein name, species, subcellular localisation and canonical sequence."""
    empty = dict(
        uniprot_id=uniprot_id, uniprot_kb_id=None,
        protein_name=None, species=None, localisation=None, canonical_seq=None
    )
    if not uniprot_id:
        return empty
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    headers = {'User-Agent': 'Python/tegel-secretome-expert'}
    try:
        resp = session.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        uniprot_kb_id = data.get('uniProtkbId')
        species = data.get('organism', {}).get('scientificName')
        rec_name = data.get('proteinDescription', {}).get('recommendedName', {})
        protein_name = rec_name.get('fullName', {}).get('value')
        locations = [
            subloc.get('location', {}).get('value')
            for comment in data.get('comments', [])
            if comment.get('commentType') == 'SUBCELLULAR LOCATION'
            for subloc in comment.get('subcellularLocations', [])
        ]
        localisation = '; '.join(sorted(set(l for l in locations if l))) or None
        canonical_seq = data.get('sequence', {}).get('value')
        return dict(
            uniprot_id=uniprot_id, uniprot_kb_id=uniprot_kb_id,
            protein_name=protein_name, species=species,
            localisation=localisation, canonical_seq=canonical_seq,
        )
    except Exception as e:
        print(f"  Error fetching {uniprot_id}: {e}")
        return empty




In [7]:

# Compute construct boundaries in the canonical UniProt sequence for each unique VC construct.
# This is used to build the protein name field T1.
# Some constructs start from the propeptide-included form and others from the mature form;
# boundaries are reported as 1-indexed positions in the full-length canonical sequence.

def build_t1_name(
    uniprot_kb_id: str | None,
    gene_name: str | None,
    uniprot_id: str | None,
    target_region: str | None,
    canonical_seq: str | None,
) -> str:
    """Build the BioStudies T1 construct name including boundaries if determinable."""
    base = uniprot_kb_id or gene_name or uniprot_id or 'UNKNOWN'
    if target_region and canonical_seq:
        bounds = find_boundaries_by_alignment(target_region, canonical_seq)
        if bounds:
            return f"CD33-{base}[{bounds[0]}-{bounds[1]}]-TEV-HPC4"
    return None




In [9]:
df_kth = (
    pl.read_excel('~/Documents/nicola_paper_2026/biostudies/EXPERT_Data from KTH to Evgeny_260527.xlsx')
    .rename({
        'Löpnummer': 'lims_id',
        'Gene.name': 'T2',
        'ENSG.id': 'ensg_id',
        'Virtual.construct': 'vc_id',
        'C1_SP_tags': 'C1',
        'C2_SP_tags': 'C2',
        'T4 UniProt ID': 'T4',
        'T5 Predicted Protein Localization': 'T5',
        'E3 Culture volume (=harvest volume)': 'E3',
        'E13 Is the protein expressed AND SECRETED? Y/N': 'E13',
        'P4 Purified? Y/N': 'P4',
        'P5 Purified protein at right size? Y/N': 'P5',
        'Q1 Final batch PURITY': 'Q1', 'Q4 (ID by MS)': 'Q4',
        'P7_ug_per_ml_harvest': 'P7',
        'E1': 'protocol_id',
        'FINAL COMMENT': 'final_comment'
    })
    .with_columns(
        T4=pl.when(pl.col.T4 == 'NA').then(None).otherwise('T4'), 
        T3=pl.lit('Homo Sapiens'),
        C3=pl.lit('N/A (license)')
    )
)
assert df_kth.select(pl.col.T4.is_null().sum()).item() > 0
assert df_kth.select(pl.col.T4.str.contains(',', literal=True).sum()).item() > 0
df_kth

lims_id,T2,ensg_id,vc_id,T4,T5,C1,C2,C2_tags,protocol_id,E3,E13,P4,P5,P7_amount_purified,P7,Q1,Q4,final_comment,State,T3,C3
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,i64,str,str
973,"""COLEC10""","""ENSG00000184374""","""VC001751""","""Q9Y6Z7""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMLDIDSRPTAEVCA…","""LDIDSRPTAEVCATHTISPGPKGDDGEKGD…","""CHO""","""29 ml""","""YES""","""YES""","""YES""",697,"""24.0344827586207""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)"""
974,"""COLEC10""","""ENSG00000184374""","""VC001751""","""Q9Y6Z7""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMLDIDSRPTAEVCA…","""LDIDSRPTAEVCATHTISPGPKGDDGEKGD…","""CHO mid""","""860 ml""","""YES""","""YES""","""YES""",8096,"""9.41395348837209""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)"""
975,"""LY96""","""ENSG00000154589""","""VC001088""","""Q9Y6Y9""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMKQYWVCNSSDASI…","""KQYWVCNSSDASISYTYCDKMQYPISINVN…","""CHO""","""35 ml""","""YES""","""YES""","""YES""",454,"""12.9714285714286""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)"""
976,"""ICOS""","""ENSG00000163600""","""VC001116""","""Q9Y6W8""","""ECD""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMGEINGSANYEMFI…","""GEINGSANYEMFIFHNGGVQILCKYPDIVQ…","""CHO""","""47 ml""","""YES""","""YES""","""YES""",679,"""14.4468085106383""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)"""
977,"""TNFRSF11A""","""ENSG00000141655""","""VC002578""","""Q9Y6Q6""","""ECD""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMIAPPCTSEKHYEH…","""IAPPCTSEKHYEHLGRCCNKCEPGKYMSSK…","""CHO""","""28 ml""","""YES""","""YES""","""YES""",571,"""20.3928571428571""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
742,"""CTRL""","""ENSG00000141086""","""VC002538""","""P40313""","""Secreted to digestive system""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMCGIPAIKPALSFS…","""CGIPAIKPALSFSQRIVNGENAVLGSWPWQ…","""CHO""","""42 ml""","""YES""","""YES""","""YES""",0,"""0""",null,"""YES""","""Failed in MS due to impurities…",4,"""Homo Sapiens""","""N/A (license)"""
743,"""CERS1""","""ENSG00000223802""","""VC000442""","""P27544""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMAAAGPAAALLQAL…","""AAAGPAAALLQALGLRDEPQGAPRLRPVPP…","""CHO""","""50 ml""","""YES""","""YES""","""YES""",0,"""0""",null,"""YES""","""Failed in MS due to impurities…",4,"""Homo Sapiens""","""N/A (license)"""
744,"""GDF1""","""ENSG00000130283""","""VC000442""","""P27539""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMAAAGPAAALLQAL…","""AAAGPAAALLQALGLRDEPQGAPRLRPVPP…","""CHO""","""50 ml""","""YES""","""YES""","""YES""",0,"""0""",null,"""YES""","""Failed in MS due to impurities…",4,"""Homo Sapiens""","""N/A (license)"""


In [10]:
assert az_df.select(pl.col.uniprot_id.n_unique().over('aa_seq').max()).item() == 1
# assert uniprot_df.select(pl.col.canonical_seq.n_unique().over('uniprot_id').max()).item() == 1
df = (
    df_kth
    .join(az_df.group_by('aa_seq').agg(uniprot_id_clean=pl.col.uniprot_id.first()), left_on='C2', right_on='aa_seq', how='left', coalesce=True)
    .with_columns(uniprot_id_clean=pl.when(pl.col.uniprot_id_clean.is_null()).then('T4').otherwise('uniprot_id_clean'))
    # .join(uniprot_df.group_by('uniprot_id').agg(pl.col.canonical_seq.first()), how='left', left_on='uniprot_id_az', right_on='uniprot_id', coalesce=True)
)
assert df.shape[0]== df_kth.shape[0]

assert df.select(pl.col.uniprot_id_clean.is_null().sum()).item() == 0
assert df.select(pl.col.uniprot_id_clean.str.contains(',', literal=True).sum()).item() == 0


In [11]:
if uniprot_cache_file.is_file():
    print(f"Reading UniProt cache from {uniprot_cache_file}")
    uniprot_df = pl.read_parquet(uniprot_cache_file)
else:
    uniprot_ids = df.select(pl.col('uniprot_id_clean').drop_nulls().unique())['uniprot_id_clean'].to_list()
    records = []
    for uid in tqdm(uniprot_ids, total=len(uniprot_ids), desc='Fetching from Uniprot REST API') :
        rec = get_uniprot_data(uid)
        records.append(rec)
    uniprot_df = pl.DataFrame(
        records,
        schema={
            'uniprot_id':    pl.String,
            'uniprot_kb_id': pl.String,
            'protein_name':  pl.String,
            'species':       pl.String,
            'localisation':  pl.String,
            'canonical_seq': pl.String,
        },
    )
    uniprot_df = uniprot_df.with_columns(pl.col.uniprot_kb_id.str.replace(r'_[^_]*$', ''))
    uniprot_df.write_parquet(uniprot_cache_file)
    print(f"Saved to {uniprot_cache_file}")

uniprot_df.head(3)
df = df.join(uniprot_df, how='left', left_on='uniprot_id_clean', right_on='uniprot_id', coalesce=True)
# assert df.select(pl.col.canonical_seq.is_null().sum()).item() == 0

Fetching from Uniprot REST API:   0%|          | 0/2127 [00:00<?, ?it/s]

Saved to /Users/evgeny/Documents/nicola_paper_2026/tegel_secretome_uniprot.parquet


In [12]:
t1_names = [
    build_t1_name(r['uniprot_kb_id'], r['T2'], r['uniprot_id_clean'],
                  r['C2'], r['canonical_seq'])
    for r in tqdm(
        df.iter_rows(named=True),
        total=df.height,
        desc='Construct boundaries',
    )
]
df = df.with_columns(
    T1=pl.Series(t1_names),
    T5=pl.when(pl.col.T5.is_null()).then('localisation').otherwise('T5')
)


Construct boundaries:   0%|          | 0/2974 [00:00<?, ?it/s]

In [13]:


blast_dir = pathlib.Path('/Users/evgeny/code/nicola_paper/secretome_blast')
blast_rows = []
for blast_file in sorted(blast_dir.glob('*.json')):
    with blast_file.open() as f:
        report = json.load(f)

    top_hit = (report.get('hits') or [None])[0]
    if top_hit is None:
        continue

    query_def = report.get('query_def')
    if query_def is None:
        continue

    hit_id = top_hit.get('hit_id')
    blast_base = None
    if hit_id:
        blast_base = hit_id.rsplit('_', 1)[0]

    hit_hsps = top_hit.get('hit_hsps')
    if  not hit_hsps:
        continue
    hsp = hit_hsps[0]
    assert hsp['hsp_num'] == 1

    blast_rows.append(
        {
            'lims_id': int(query_def),
            'blast_hit_acc': top_hit.get('hit_acc'),
            'blast_hit_id': hit_id,
            'blast_base': blast_base or top_hit.get('hit_uni_gn') or top_hit.get('hit_acc'),
            'blast_hit_from': hsp['hsp_hit_from'],
            'blast_hit_to': hsp['hsp_hit_to'],
            'blast_T1': f"CD33-{blast_base}[{hsp['hsp_hit_from']}-{hsp['hsp_hit_to']}]-TEV-HPC4"
        }
    )

blast_df = pl.DataFrame(blast_rows).join(df.select('lims_id', 'C2'), on='lims_id', how='inner')

print("Before backfill from BLAST:", df.select(pl.col.T1.is_null().sum()).item(), "rows with missing T1")

df = (df
    .join(blast_df.select('C2', 'blast_T1'), on='C2', how='left')
    .with_columns(T1=pl.when(pl.col.T1.is_null()).then('blast_T1').otherwise('T1'))
)

print("After backfill from BLAST:", df.select(pl.col.T1.is_null().sum()).item(), "rows with missing T1")




Before backfill from BLAST: 16 rows with missing T1
After backfill from BLAST: 0 rows with missing T1


In [14]:
df.filter(pl.col.blast_T1.is_not_null())

lims_id,T2,ensg_id,vc_id,T4,T5,C1,C2,C2_tags,protocol_id,E3,E13,P4,P5,P7_amount_purified,P7,Q1,Q4,final_comment,State,T3,C3,uniprot_id_clean,uniprot_kb_id,protein_name,species,localisation,canonical_seq,T1,blast_T1
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str
2250,"""ELN""","""ENSG00000049540""","""VC000383""","""P15502""","""Secreted to extracellular matr…","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMGGVPGAIPGGVPG…","""GGVPGAIPGGVPGGVFYPGAGLGALGGGAL…","""HEK expi""","""39 ml""","""YES""","""YES""","""YES""",418,"""10.7179487179487""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""F8WAH6""","""F8WAH6""",null,null,null,null,"""CD33-ELN[27-786]-TEV-HPC4""","""CD33-ELN[27-786]-TEV-HPC4"""
2545,"""AMY1A""","""ENSG00000237763""","""VC001430""","""P04745""","""Secreted to digestive system""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMQYSSNTQQGRTSI…","""QYSSNTQQGRTSIVHLFEWRWVDIALECER…","""CHO""","""27 ml""","""YES""","""YES""","""YES""",2265,"""83.8888888888889""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""P04745""","""AMY1A""",null,null,null,null,"""CD33-AMY1C[16-511]-TEV-HPC4""","""CD33-AMY1C[16-511]-TEV-HPC4"""
2546,"""AMY1A""","""ENSG00000237763""","""VC001430""","""P04745""","""Secreted to digestive system""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMQYSSNTQQGRTSI…","""QYSSNTQQGRTSIVHLFEWRWVDIALECER…","""CHO mid""","""850 ml""","""YES""","""YES""","""YES""",78090,"""91.8705882352941""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""P04745""","""AMY1A""",null,null,null,null,"""CD33-AMY1C[16-511]-TEV-HPC4""","""CD33-AMY1C[16-511]-TEV-HPC4"""
2547,"""AMY1B""","""ENSG00000174876""","""VC001430""","""P04745""","""Secreted to digestive system""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMQYSSNTQQGRTSI…","""QYSSNTQQGRTSIVHLFEWRWVDIALECER…","""CHO""","""27 ml""","""YES""","""YES""","""YES""",2265,"""83.8888888888889""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""P04745""","""AMY1A""",null,null,null,null,"""CD33-AMY1C[16-511]-TEV-HPC4""","""CD33-AMY1C[16-511]-TEV-HPC4"""
2548,"""AMY1B""","""ENSG00000174876""","""VC001430""","""P04745""","""Secreted to digestive system""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMQYSSNTQQGRTSI…","""QYSSNTQQGRTSIVHLFEWRWVDIALECER…","""CHO mid""","""850 ml""","""YES""","""YES""","""YES""",78090,"""91.8705882352941""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""P04745""","""AMY1A""",null,null,null,null,"""CD33-AMY1C[16-511]-TEV-HPC4""","""CD33-AMY1C[16-511]-TEV-HPC4"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2973,"""IFNA1""","""ENSG00000197919""","""VC002347""","""P01562""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMGCDLPETHSLDNR…","""GCDLPETHSLDNRRTLMLLAQMSRISPSSC…","""CHO pilot""","""10000 ml""","""YES""","""YES""","""YES""",544000,"""54.4""",""">80%""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""P01562""","""IFNA1""",null,null,null,null,"""CD33-IFNA1[23-189]-TEV-HPC4""","""CD33-IFNA1[23-189]-TEV-HPC4"""
274,"""C22orf46""","""ENSG00000184208""","""VC002255""","""C9J442""",null,"""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMWEPVQGLLSQNHS…","""WEPVQGLLSQNHSCRDPQCCGNLLVLCLFL…","""CHO""","""40 ml""","""NO""","""NO""","""NO""",0,"""0""",null,"""NO""","""Fail no/low production = no pu…",2,"""Homo Sapiens""","""N/A (license)""","""C9J442""","""CV046""",null,null,null,null,"""CD33-A0A2I3SK80[20-233]-TEV-HP…","""CD33-A0A2I3SK80[20-233]-TEV-HP…"
408,"""C9orf47""","""ENSG00000186354""","""VC002323""","""Q6ZRZ4""",null,"""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMQAPAQAQTSDLVP…","""QAPAQAQTSDLVPSLFPLGLWAPGFCTWSS…","""CHO""","""40 ml""","""NO""","""NO""","""NO""

In [16]:
kth_experimental_conditions_df = (
    pl.read_excel('~/Documents/nicola_paper_2026/biostudies/tegel_secretome_expert_corrected.xlsx')[4:8]
    .with_columns(pl.col.ID.replace({
        'CHO small-scale': 'CHO',
        'CHO mid-scale': 'CHO mid',
        'CHO pilot-scale': 'CHO pilot',
        'HEK small-scale': 'HEK expi'
    })).rename({'ID': 'protocol_id'})
    .drop('T1', 'T2', 'T3', 'T4','T5', 'C1', 'C2', 'C3', 'E3', 'P7', 'P4', 'P5', 'E13', 'Q4')
)
kth_experimental_conditions_df

protocol_id,T6,T7,CX1,CX2,CX3,C4,C5,C6,E1,E2,E4,E5,E6,E7,E8,E9,E10,E11,E12,EM1,EM2,EM3,EM4,EM5,P1,P2,P3,P6,P8,P9,P10,P11,P12,P13,P14,P15,Q1,Q2,Q3,O1,O2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CHO""",null,null,null,null,null,"""pQMCF-1-MCS (Icosagen Cell Fac…",null,null,"""CHO cells, QMCF Technology, Ic…","""Mix of CD CHO Medium, 293 SFM …","""37°C => 30°C""","""Feed: Feed B (Gibco)""","""110 rpm, orbital 25 mm""","""125 ml Cornig shake flasks, ve…","""13 d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""BTX electroporation (twin wave…",""" """,null,null,null,"""Protein C-tag (HPC4) affinity …","""pH 7.5; BUFF Tris, 20 mM; SALT…",null,"""spectrophotometry at 280nm, DS…",null,null,null,null,null,null,null,"""pH 7.4; BUFF Na2HPO4, 8mM; BUF…","""SDS-PAGE, WB at step 1""",null,null,"""VC000116, small""","""8"""
"""CHO mid""",null,null,null,null,null,"""pQMCF-1-MCS (Icosagen Cell Fac…",null,null,"""CHO cells, QMCF Technology, Ic…","""CHO TF, Glutamax ""","""37°C => 30°C""","""Feed: Basic feed (Xell)""","""110-140 rpm, orbital 25 mm""","""1,6-2,8 L Optimum Growth flask…","""13 d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""Reagent 007 (Icosagen Cell Fac…",""" """,null,null,null,"""Protein C-tag (HPC4) affinity …","""pH 7.5; BUFF Tris, 20 mM; SALT…",null,"""spectrophotometry at 280nm, DS…",null,null,null,null,null,null,null,"""pH 7.4; BUFF Na2HPO4, 8mM; BUF…","""SDS-PAGE, WB at step 1""",null,null,"""VC000116, medium""","""8"""
"""CHO pilot""",null,null,null,null,null,"""pQMCF-1-MCS (Icosagen Cell Fac…",null,null,"""CHO cells, QMCF Technology, Ic…","""Mix of CD CHO Medium, 293 SFM …","""37°C => 30°C""","""Feed: Feed B (Gibco)""","""NA - Wave Bioreactor""","""20 L Cellbag (GE Healthcare; W…","""27 - 28 d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""BTX electroporation (twin wave…",""" """,null,null,null,"""Protein C-tag (HPC4) affinity …","""pH 7.5; BUFF Tris, 20 mM; SALT…",null,"""spectrophotometry at 280nm, DS…",null,null,null,null,null,null,null,"""pH 7.4; BUFF Na2HPO4, 8mM; BUF…","""SDS-PAGE, WB at step 1""",null,null,"""VC000116, pilot""","""8"""
"""HEK expi""",null,null,null,null,null,"""pKTH16e""",null,null,"""Expi293F, Thermo Fisher Scient…","""Expi293 Expression Media""","""37°C""",null,"""110 rpm, orbital 25 mm""","""125 ml Cornig shake flasks, ve…","""4d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""30 ug DNA to 0,12 mg PEI MAX (…",""" """,null,null,null,"""Protein C-tag (HPC4) affinity …","""pH 7.5; BUFF Tris, 20 mM; SALT…",null,"""spectrophotometry at 280nm, DS…",null,null,null,null,null,null,null,"""pH 7.4; BUFF Na2HPO4, 8mM; BUF…","""SDS-PAGE, WB at step 1""",null,null,"""VC000010, HEK""","""8"""


In [23]:
secretome_df_raw = df.join(kth_experimental_conditions_df, on='protocol_id')
secretome_df_raw = secretome_df_raw.with_columns(
    pl.col.Q1 + pl.lit(', ') + pl.col.Q1_right,
    pl.col.P7 + pl.lit(' ug/ml'),
    O1=pl.format('{{"ENSG ID":"{ensg_id}", "final comment": "{final_comment}", "virtual construct id": "{vc_id}", "amount purified": "{P7_amount_purified}"}}')
)
secretome_df_raw.write_parquet("~/Documents/nicola_paper_2026/biostudies/secretome_df_raw.parquet")
secretome_df_raw

lims_id,T2,ensg_id,vc_id,T4,T5,C1,C2,C2_tags,protocol_id,E3,E13,P4,P5,P7_amount_purified,P7,Q1,Q4,final_comment,State,T3,C3,uniprot_id_clean,uniprot_kb_id,protein_name,species,localisation,canonical_seq,T1,blast_T1,T6,T7,CX1,CX2,CX3,C4,C5,C6,E1,E2,E4,E5,E6,E7,E8,E9,E10,E11,E12,EM1,EM2,EM3,EM4,EM5,P1,P2,P3,P6,P8,P9,P10,P11,P12,P13,P14,P15,Q1_right,Q2,Q3,O1,O2
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
973,"""COLEC10""","""ENSG00000184374""","""VC001751""","""Q9Y6Z7""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMLDIDSRPTAEVCA…","""LDIDSRPTAEVCATHTISPGPKGDDGEKGD…","""CHO""","""29 ml""","""YES""","""YES""","""YES""",697,"""24.0344827586207 ug/ml""",""">80%, SDS-PAGE, WB at step 1""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""Q9Y6Z7""","""COL10""","""Collectin-10""","""Homo sapiens""","""Cytoplasm; Golgi apparatus; Se…","""MNGFASLLRRNQFILLVLFLLQIQSLGLDI…","""CD33-COL10[14-277]-TEV-HPC4""",null,null,null,null,null,null,"""pQMCF-1-MCS (Icosagen Cell Fac…",null,null,"""CHO cells, QMCF Technology, Ic…","""Mix of CD CHO Medium, 293 SFM …","""37°C => 30°C""","""Feed: Feed B (Gibco)""","""110 rpm, orbital 25 mm""","""125 ml Cornig shake flasks, ve…","""13 d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""BTX electroporation (twin wave…",""" """,null,null,null,"""Protein C-tag (HPC4) affinity …","""pH 7.5; BUFF Tris, 20 mM; SALT…",null,"""spectrophotometry at 280nm, DS…",null,null,null,null,null,null,null,"""pH 7.4; BUFF Na2HPO4, 8mM; BUF…","""SDS-PAGE, WB at step 1""",null,null,"""{""ENSG ID"":""ENSG00000184374"", …","""8"""
974,"""COLEC10""","""ENSG00000184374""","""VC001751""","""Q9Y6Z7""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMLDIDSRPTAEVCA…","""LDIDSRPTAEVCATHTISPGPKGDDGEKGD…","""CHO mid""","""860 ml""","""YES""","""YES""","""YES""",8096,"""9.41395348837209 ug/ml""",""">80%, SDS-PAGE, WB at step 1""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""Q9Y6Z7""","""COL10""","""Collectin-10""","""Homo sapiens""","""Cytoplasm; Golgi apparatus; Se…","""MNGFASLLRRNQFILLVLFLLQIQSLGLDI…","""CD33-COL10[14-277]-TEV-HPC4""",null,null,null,null,null,null,"""pQMCF-1-MCS (Icosagen Cell Fac…",null,null,"""CHO cells, QMCF Technology, Ic…","""CHO TF, Glutamax ""","""37°C => 30°C""","""Feed: Basic feed (Xell)""","""110-140 rpm, orbital 25 mm""","""1,6-2,8 L Optimum Growth flask…","""13 d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""Reagent 007 (Icosagen Cell Fac…",""" """,null,null,null,"""Protein C-tag (HPC4) affinity …","""pH 7.5; BUFF Tris, 20 mM; SALT…",null,"""spectrophotometry at 280nm, DS…",null,null,null,null,null,null,null,"""pH 7.4; BUFF Na2HPO4, 8mM; BUF…","""SDS-PAGE, WB at step 1""",null,null,"""{""ENSG ID"":""ENSG00000184374"", …","""8"""
975,"""LY96""","""ENSG00000154589""","""VC001088""","""Q9Y6Y9""","""Secreted to blood""","""ATGCCCCTGCTGCTGCTGCTGCCCCTGCTG…","""MPLLLLLPLLWAGALAMKQYWVCNSSDASI…","""KQYWVCNSSDASISYTYCDKMQYPISINVN…","""CHO""","""35 ml""","""YES""","""YES""","""YES""",454,"""12.9714285714286 ug/ml""",""">80%, SDS-PAGE, WB at step 1""","""YES""","""Finished protein""",1,"""Homo Sapiens""","""N/A (license)""","""Q9Y6Y9""","""LY96""","""Lymphocyte antigen 96""","""Homo sapiens""","""Secreted; Secreted, extracellu…","""MLPFLFFSTLFSSIFTEAQKQYWVCNSSDA…","""CD33-LY96[1-160]-TEV-HPC4""",null,null,null,null,null,null,"""pQMCF-1-MCS (Icosagen Cell Fac…",null,null,"""CHO cells, QMCF Technology, Ic…","""Mix of CD CHO Medium, 293 SFM …","""37°C => 30°C""","""Feed: Feed B (Gibco)""","""110 rpm, orbital 25 mm""","""125 ml Cornig shake flasks, ve…","""13 d""",null,null,null,"""WB with GTX18591 ab, SDS-PAGE""","""BTX electroporation (twin wave…","""

In [19]:
supp_sheet_file = pathlib.Path(
    '~/Documents/nicola_paper_2026/Supplementary Sheet S2_26052026.xlsx'
).expanduser()
expert_metadata = (
    pl.read_excel(supp_sheet_file, sheet_name='Mammalian cells', has_header=False)
    .transpose(column_names='column_1')
    .select('ID', 'Description', 'Importance', 'What to capture')
)
secretome_df = pl.concat([
    expert_metadata.transpose(column_names='ID'), 
    secretome_df_raw
], how='diagonal').select(expert_metadata['ID'].to_list())
secretome_df
# df

T1,T2,T3,T4,T5,T6,T7,CX1,CX2,CX3,C1,C2,C3,C4,C5,C6,E1,E2,E3,E4,E5,E6,E7,E8,E9,E10,E11,E12,E13,EM1,EM2,EM3,EM4,EM5,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,Q1,Q2,Q3,Q4,O1,O2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
""" Protein name (designed const…","""Gene name (HGNC) ""","""Gene species (NCBI)""","""UniProt ID ""","""Predicted protein localisation…","""PDB id""","""PTMs""","""Complex ID ""","""Complex name ""","""Complex formation method ""","""DNA sequence of coding region …","""Amino acid sequence including …","""Nucleotide sequence transcript""","""Vector name ""","""Vector sequence ""","""Annotated sequence""","""Host strain or cell line""","""Culture medium name or source""","""Culture volume ""","""Culture conditions - temperatu…","""Culture conditions-additives""","""Culture conditions - rpm""","""Culture conditions - type of g…","""Growth time ""","""Pellet weight ""","""Lysis method ""","""Lysis buffer""","""Total expression determination…","""Total expression - is the prot…","""Culture conditions - transfect…","""Virus generation (if BacMam)""","""Culture conditions - BacMam tr…","""Stable cell line generation""","""Cell density at transfection o…","""First-step purification method""","""First-step purification buffer…","""First-step purification binary…","""First-step binary purification…","""First-step purification protei…","""First-step purification yield …","""First-step purification yield ""","""Second-step purification metho…","""Second-step purification yield…","""Second-step purification yield…","""Third-step purification method…","""Third-step purification yield …","""Third-step purification yield ""","""Tag removal step""","""Final protein buffer""","""Final batch - purity ""","""Final batch - aggregation stat…","""Final batch - stability / ther…","""Final batch - identity and-or …","""Other comments""","""Data record completeness score"""
"""Critical""","""Highly Enabling""","""Optional""","""Optional""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Mandatory for complexes""","""Mandatory for complexes""","""Highly Enabling for Complexes""","""Critical""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Optional""","""Critical""","""Critical""","""Critical""","""Critical""","""Optional""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Optional""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Optional""","""Optional""","""Highly Enabling""","""Optional""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Critical""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Optional""","""Highly Enabling""","""Optional""","""Optional""","""Highly Enabling""","""Highly Enabling""","""Highly Enabling""","""Optional""","""Optional""","""Optional""","""Highly Enabling"""
"""Protein name according to UniP…","""Gene name according to HGNC na…","""Taxonomy according to NCBI (e.…","""UniProt identifier e.g. P00918""","""Intracellular, membrane-bound,…","""PDB identifier""","""Mapped post-translational modi…","""An id to link different record…","""Descriptive name of the comple…","""Polycistronic construct, or mu…","""Nucleotide sequence from start…","""Amino acid sequence from start…","""Nucleotide sequence from trans…","""Vector name excluding insert (…","""Nucleotide sequence of entire …","""Vector or insert sequence with…","""Name of mammalian strain used …","""Name and composition of medium…","""ml to L""","""Temperature used (e.g. 37°C)""","""Any additional co-factors or c…","""Shaker frequency used during g…","""Type and volume of growth cont…","""Hours to days""","""Wet weight of pellet in g""","""How cells were

In [ ]:
# secretome_df.write_excel('~/Documents/nicola_paper_2026/secretome_expert.xlsx')

In [20]:
secretome_df_raw.select(pl.col.C1.n_unique())

C1
u32
2167


In [21]:
secretome_df_raw.shape

(2974, 71)

In [22]:
(secretome_df_raw['P4'] == 'YES').mean()

0.7652992602555481

In [26]:
secretome_df_raw.select('T1', 'C2')

T1,C2
str,str
"""CD33-COL10[14-277]-TEV-HPC4""","""MPLLLLLPLLWAGALAMLDIDSRPTAEVCA…"
"""CD33-COL10[14-277]-TEV-HPC4""","""MPLLLLLPLLWAGALAMLDIDSRPTAEVCA…"
"""CD33-LY96[1-160]-TEV-HPC4""","""MPLLLLLPLLWAGALAMKQYWVCNSSDASI…"
"""CD33-ICOS[20-141]-TEV-HPC4""","""MPLLLLLPLLWAGALAMGEINGSANYEMFI…"
"""CD33-TNR11[14-212]-TEV-HPC4""","""MPLLLLLPLLWAGALAMIAPPCTSEKHYEH…"
…,…
"""CD33-CTRL[4-264]-TEV-HPC4""","""MPLLLLLPLLWAGALAMCGIPAIKPALSFS…"
"""CD33-GDF1[13-372]-TEV-HPC4""","""MPLLLLLPLLWAGALAMAAAGPAAALLQAL…"
"""CD33-GDF1[13-372]-TEV-HPC4""","""MPLLLLLPLLWAGALAMAAAGPAAALLQAL…"


In [ ]:
assert secretome_df_raw.group_by('C2').agg(pl.col.T1.n_unique()).max().select('T1').item() == 1
fasta_df = (
    secretome_df_raw.with_row_index("row_idx")
    .group_by('C2')
    .agg(fasta_id=pl.col.T1.first() + pl.lit('-') + (pl.col.row_idx.first() + 1).cast(pl.String))
)
import typing, gzip, os
def resolve(filename) -> pathlib.Path:
    if type(filename) == str:
        filename = pathlib.Path(filename)
    if str(filename).startswith('~'):
        filename = filename.expanduser()
    return filename.resolve()

def write_fasta(sequences:typing.Iterable[typing.Tuple[str, str]]|typing.Mapping[str, str], filename:str|os.PathLike, seq_chunk_size:int=80):
    """
    Write a sequence of (id, sequence) tuples to a FASTA file.
    :param sequences: A sequence of (id, sequence) tuples.
    :param filename: The name of the file to write to.
    :param seq_chunk_size: The size of each sequence chunk in the output file.
    """
    if isinstance(sequences, typing.Mapping):
        sequences = sequences.items()
    filename = resolve(filename)
    is_gzip = filename.suffix == '.gz'
    with gzip.open(filename, 'wt') if is_gzip else open(filename, 'w') as f:
        for seq_id, sequence in sequences:
            f.write(f">{seq_id}\n")
            for i in range(0, len(sequence), seq_chunk_size):
                f.write(sequence[i:i + seq_chunk_size] + '\n')

In [ ]:
write_fasta(zip(fasta_df['fasta_id'], fasta_df['C2']), 
            '/Users/evgeny/Documents/nicola_paper_2026/biostudies/secretome_expert.fasta.gz')
